In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split
from keras.preprocessing.text import Tokenizer
from sklearn.preprocessing import LabelEncoder
from keras.models import Sequential
from keras.layers import Dense,Dropout
from keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from keras.utils import to_categorical
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
from sklearn.model_selection import GridSearchCV
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from scikeras.wrappers import KerasClassifier


In [18]:
df = pd.read_csv("epc_best20.csv")

In [19]:
df.head()

,Unnamed: 0,Total floor area (m²),Total current energy costs over 3 years (£),Current hot water costs over 3 years (£),Part 1 Construction Age Band,Part 1 Floor 0 Room Height,Low Energy Lighting %,Mechanical Ventilation,Tenure,Transaction Type,...,insulated_wall,wall_type,roof_type,floor_type,windows_glazing,MMH_mains gas,MHCS_programmer,low_lighting,SH_room heaters,Current energy efficiency rating band
0,0,45.0,3771.0,1176.0,before 1919,3.10,100,natural,owner-occupied,none of the above,...,0,sandstone or limestone,Pitched,Unknown,double glazed,False,True,low energy lighting 100% of fixed outlets,False,E
1,1,143.0,2793.0,375.0,2003-2007,2.40,100,natural,owner-occupied,marketed sale,...,1,cavity wall,Pitched,Suspended,double glazed,True,True,low energy lighting 100% of fixed outlets,False,C
2,2,65.0,1947.0,441.0,before 1919,2.45,100,natural,owner-occupied,marketed sale,...,0,sandstone or limestone,Unknown,Solid,single glazed,True,True,low energy lighting 100% of fixed outlets,False,C
3,3,49.0,1158.0,195.0,1999-2002,2.40,100,natural,rented (private),none of the above,...,1,cavity wall,Unknown,Suspended,double glazed,True,True,low energy lighting 100% of fixed outlets,False,C
4,4,212.0,6477.0,819.0,before 1919,2.80,64,natural,owner-occupied,marketed sale,...,1,granite or whinstone,Pitched,Solid,double glazed,False,True,low energy lighting 60% of fixed outlets,True,E


In [20]:
df.tail()

,Unnamed: 0,Total floor area (m²),Total current energy costs over 3 years (£),Current hot water costs over 3 years (£),Part 1 Construction Age Band,Part 1 Floor 0 Room Height,Low Energy Lighting %,Mechanical Ventilation,Tenure,Transaction Type,...,insulated_wall,wall_type,roof_type,floor_type,windows_glazing,MMH_mains gas,MHCS_programmer,low_lighting,SH_room heaters,Current energy efficiency rating band
54490,54490,74.0,996.0,210.0,before 1919,2.40,100,natural,owner-occupied,new dwelling,...,0,Unknown,Unknown,Unknown,high performance glazing,True,True,low energy lighting 100% of fixed outlets,False,B
54491,54491,228.0,4674.0,1491.0,2008 onwards,2.40,100,natural,owner-occupied,none of the above,...,1,timber frame,Pitched,Solid,double glazed,False,False,low energy lighting 100% of fixed outlets,False,C
54492,54492,123.0,2292.0,753.0,before 1919,2.43,100,"mechanical, supply and extract",unknown,new dwelling,...,0,Unknown,Unknown,Unknown,high performance glazing,False,False,low energy lighting 100% of fixed outlets,False,B
54493,54493,59.0,972.0,276.0,before 1919,2.41,100,natural,rented (social),new dwelling,...,0,Unknown,Unknown,Unknown,high performance glazing,True,False,low energy lighting 100% of fixed outlets,False,B
54494,54494,77.0,978.0,303.0,before 1919,2.40,100,natural,rented (social),new dwelling,...,0,Unknown,Unknown,Unknown,high performance glazing,True,False,low energy lighting 100% of fixed outlets,False,B


In [21]:
# Define which columns are categorical and which are numerical
categorical_cols = ["Part 1 Construction Age Band", "Mechanical Ventilation", "Tenure", "Transaction Type",
                   "wall_insulation", "roof_insulation", "wall_type", "roof_type", "floor_type",
                   "windows_glazing", "low_lighting"]
numerical_cols = ["Total floor area (m²)", "Total current energy costs over 3 years (£)",
                  "Current hot water costs over 3 years (£)", "Part 1 Floor 0 Room Height",
                  "Low Energy Lighting %", "insulated_wall", "MMH_mains gas", "MHCS_programmer",
                  "SH_room heaters"]

In [22]:
# Create ColumnTransformer to handle different data types
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(), categorical_cols)])

In [23]:
# Split data into features and labels
X = df.drop("Current energy efficiency rating band", axis=1)
y = df["Current energy efficiency rating band"]
encoder = LabelEncoder()
y= encoder.fit_transform(y)

# One-hot encode labels
y = to_categorical(y, num_classes=7)

In [24]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [25]:
# Fit and transform the data using the ColumnTransformer
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [26]:
# fix random seed for reproducibility
seed = 7
tf.random.set_seed(seed)

In [27]:
# Function to create model, required for KerasClassifier
def create_model():
    model = Sequential()
    model.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
    model.add(Dropout(0.5))  # Adding dropout to prevent overfitting
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.5))  # Adding dropout to prevent overfitting
    model.add(Dense(7, activation='softmax'))

    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

In [28]:
model = KerasClassifier(model=create_model, verbose=0)

# Tune Batch Size and Number of Epochs

In [13]:
# define the grid search parameters
batch_size = [10, 20, 40, 60, 80, 100]
epochs = [20, 50, 100]
param_grid = dict(batch_size=batch_size, epochs=epochs)
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)
grid_result = grid.fit(X_train, y_train)

C:\Users\harth\anaconda3\envs\tf\lib\site-packages\tensorflow\python\framework\indexed_slices.py:446: UserWarning: Converting sparse IndexedSlices(IndexedSlices(indices=Tensor("gradient_tape/sequential/dense/embedding_lookup_sparse/Reshape_1:0", shape=(None,), dtype=int32), values=Tensor("gradient_tape/sequential/dense/embedding_lookup_sparse/Reshape:0", shape=(None, 64), dtype=float32), dense_shape=Tensor("gradient_tape/sequential/dense/embedding_lookup_sparse/Cast:0", shape=(2,), dtype=int32))) to a dense Tensor of unknown shape. This may consume a large amount of memory.
  "shape. This may consume a large amount of memory." % value)


In [14]:
# summarize results
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))

Best: 0.862487 using {'batch_size': 80, 'epochs': 100}
0.851225 (0.000536) with: {'batch_size': 10, 'epochs': 20}
0.856317 (0.002434) with: {'batch_size': 10, 'epochs': 50}
0.858450 (0.004527) with: {'batch_size': 10, 'epochs': 100}
0.849642 (0.004118) with: {'batch_size': 20, 'epochs': 20}
0.857097 (0.002855) with: {'batch_size': 20, 'epochs': 50}
0.860469 (0.001373) with: {'batch_size': 20, 'epochs': 100}
0.854184 (0.001784) with: {'batch_size': 40, 'epochs': 20}
0.855170 (0.000361) with: {'batch_size': 40, 'epochs': 50}
0.861868 (0.002255) with: {'batch_size': 40, 'epochs': 100}
0.849573 (0.001511) with: {'batch_size': 60, 'epochs': 20}
0.857074 (0.002486) with: {'batch_size': 60, 'epochs': 50}
0.860928 (0.002973) with: {'batch_size': 60, 'epochs': 100}
0.850101 (0.002563) with: {'batch_size': 80, 'epochs': 20}
0.857074 (0.004249) with: {'batch_size': 80, 'epochs': 50}
0.862487 (0.002112) with: {'batch_size': 80, 'epochs': 100}
0.849046 (0.002307) with: {'batch_size': 100, 'epochs':

In [29]:
# define the grid search parameters
optimizer = ['SGD', 'RMSprop', 'Adagrad', 'Adadelta', 'Adam', 'Adamax', 'Nadam']
param_grid = dict(optimizer=optimizer)
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)
grid_result = grid.fit(X_train, y_train)

C:\Users\harth\anaconda3\envs\tf\lib\site-packages\tensorflow\python\framework\indexed_slices.py:446: UserWarning: Converting sparse IndexedSlices(IndexedSlices(indices=Tensor("gradient_tape/sequential_1/dense_3/embedding_lookup_sparse/Reshape_1:0", shape=(None,), dtype=int32), values=Tensor("gradient_tape/sequential_1/dense_3/embedding_lookup_sparse/Reshape:0", shape=(None, 64), dtype=float32), dense_shape=Tensor("gradient_tape/sequential_1/dense_3/embedding_lookup_sparse/Cast:0", shape=(2,), dtype=int32))) to a dense Tensor of unknown shape. This may consume a large amount of memory.
  "shape. This may consume a large amount of memory." % value)


In [30]:
# summarize results
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))

Best: 0.754152 using {'optimizer': 'Adagrad'}
0.752386 (0.011345) with: {'optimizer': 'SGD'}
0.751422 (0.004091) with: {'optimizer': 'RMSprop'}
0.754152 (0.003422) with: {'optimizer': 'Adagrad'}
0.751399 (0.002829) with: {'optimizer': 'Adadelta'}
0.749404 (0.000941) with: {'optimizer': 'Adam'}
0.753349 (0.011862) with: {'optimizer': 'Adamax'}
0.752156 (0.010053) with: {'optimizer': 'Nadam'}
